# ChakraModel R4: E3 extra seeds (43, 44)
Adds two more seeds to the ClinicDB leakage study, so E3 matches the 3-seed protocol of every other experiment. 40 runs: 5 folds x {video-grouped, random} x {U-Net-R34, SegFormer-B2}.

**Before you run, set:**
- **Settings:** Accelerator = **GPU T4 x2**, Internet = **On**.
- **Inputs (Add Input), only these two datasets:** chakra-leakbench-v1, chakra-leakbench-r2-v1-22-09-14-50
- **Run:** Save Version -> Save & Run All (Commit).

**Split it over two accounts if quota is tight:** set `HALF = "A"` (jobs 50-69) in one account and `HALF = "B"` (jobs 70-89) in another; each takes about 3.5 h. Leave `HALF = ""` to run all 40 in one go (about 7 h).

**Expected log:** `DATA ...`, `CODE ...`, `jobs 40 50 - 89`, then `=== job 50 ...` lines. Ends with `[0, 0]` and `finished runs 40` (or 20 per half). Re-committing skips runs that already finished.

In [1]:
import subprocess
subprocess.run('pip -q install segmentation-models-pytorch==0.5.0 timm', shell=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.2 MB/s eta 0:00:00


CompletedProcess(args='pip -q install segmentation-models-pytorch==0.5.0 timm', returncode=0)

In [2]:
# E3 extra seeds (43, 44): 40 runs = 5 folds x {grouped, random} x {U-Net, SegFormer}.
# Inputs: chakra-leakbench-v1 (Stage 1 data) + chakra-leakbench-r2-... (patched code + manifests). No other inputs.
# HALF = "" runs all 40 across both GPUs (~7 h). HALF = "A" runs jobs 50-69, "B" runs 70-89 (~3.5 h each, two accounts).
HALF = ""
import glob, os, csv, subprocess
IN = "/kaggle/input"
def find_dir(marker, depth=6):
    for d in range(1, depth + 1):
        h = glob.glob(os.path.join(IN, *(["*"] * d), marker))
        if h: return os.path.dirname(h[0])
    raise FileNotFoundError(marker)
ROOT = find_dir("fixed_polyp_dataset_v1"); R2 = find_dir("runs_e1"); W = "/kaggle/working/leakbench"
print("DATA", ROOT, "\nCODE", R2, flush=True)
subprocess.run(f"rm -rf {W} && cp -r {R2}/leakbench {W}", shell=True, check=True)
jobs = [dict(job=50 + i, experiment="E3_leak", manifest=f"manifests/leak_f{k}_{mode}.csv", model=m, seed=s, epochs=50)
        for i, (s, k, mode, m) in enumerate((s, k, mode, m) for s in (43, 44) for k in range(5)
                                            for mode in ("grouped", "random") for m in ("unet_r34", "segformer_b2"))]
with open(f"{W}/jobs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(jobs[0])); w.writeheader(); w.writerows(jobs)
assert all(os.path.exists(f"{W}/{j['manifest']}") for j in jobs); print("jobs", len(jobs), jobs[0]["job"], "-", jobs[-1]["job"], flush=True)
SL = {"": ("50-69", "70-89"), "A": ("50-59", "60-69"), "B": ("70-79", "80-89")}[HALF]
cmd = lambda g: (f"CUDA_VISIBLE_DEVICES={g} python {W}/run_queue.py --data-root {ROOT} --jobs {SL[g]} "
                 f"--out /kaggle/working/runs_e3seeds --budget-hours 11.0 >> /kaggle/working/log_r4_gpu{g}.txt 2>&1")
procs = [subprocess.Popen(cmd(g), shell=True) for g in (0, 1)]; print([p.wait() for p in procs])
done = glob.glob("/kaggle/working/runs_e3seeds/*/results.json"); print("finished runs", len(done))
subprocess.run(f"python {W}/aggregate.py /kaggle/working/runs_e3seeds", shell=True)


DATA /kaggle/input/datasets/gokulrocky/chakra-leakbench-v1 
CODE /kaggle/input/datasets/gokulrocky/chakra-leakbench-r2-v1-22-09-14-50
jobs 40 50 - 89
[0, 0]
finished runs 40
E3 segformer_b2: random-split minus video-grouped ClinicDB Dice = +0.059 (folds=5, per-fold [0.011, 0.007, 0.007, 0.103, 0.167])
E3 unet_r34: random-split minus video-grouped ClinicDB Dice = +0.070 (folds=5, per-fold [0.036, -0.001, 0.017, 0.135, 0.165])


CompletedProcess(args='python /kaggle/working/leakbench/aggregate.py /kaggle/working/runs_e3seeds', returncode=0)